In [ ]:
%pip install pydicom
%pip install wget
%pip install open_clip_torch
%pip install --upgrade transformers

In [ ]:
from modules.utils.constants import SIZE_TRANSFORM
from modules.models.factory import ModelFactory


In [ ]:
from torch.utils.data import Dataset
from torchvision import transforms
import csv
import os
from PIL import Image
_toTensor = transforms.ToTensor()

MIMIC_CLASS_NAMES = ['Phneumonia', 'Normal']
MIMIC_CLASS_TO_ID = {name: idx for idx, name in enumerate(MIMIC_CLASS_NAMES)}
ENTREP_CLASS_NAMES = ['vocal-throat', 'nose', 'ear', 'throat']
ENTREP_CLASS_TO_ID = {name: idx for idx, name in enumerate(ENTREP_CLASS_NAMES)}

class MiMic(Dataset):
    def __init__(self, size_transform):
        from datasets import load_dataset
        self.ds = load_dataset(
            "itsanmolgupta/mimic-cxr-dataset",
            split="train",
            streaming=False
        )

        self.size_transform = size_transform
        self.samples = []

        for i in range(len(self.ds)):
            if self.ds[i]["findings"]:
                self.samples.append((i, "findings"))
            if self.ds[i]["impression"]:
                self.samples.append((i, "impression"))

    def _extract_class_id(self, sample, text):
        pass

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ds_idx, field = self.samples[idx]

        sample = self.ds[ds_idx]
        img = sample["image"]
        text = sample[field]
        class_id = self._extract_class_id(sample, text)

        img = self.size_transform(img).convert("RGB")
        img = _toTensor(img)

        return {
            "image": img,
            "text": text,
            "class_id": class_id,
            "idx": idx
        }



class ENTREPDataset(Dataset):
    def __init__(
        self,
        size_transform,
        img_dir="/kaggle/input/entrep-train-contrastive/Dataset/images",
        annotation_file="/kaggle/input/entrep-train-contrastive/Dataset/data.csv",
    ):
        self.img_dir = img_dir
        self.annotation_file = annotation_file
        self.size_transform = size_transform

        self.samples = []

        with open(self.annotation_file, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                path = row["path"].replace('image', "Image")
                caption = row["caption"]
                class_id = self._extract_class_id(row)

                img_path = os.path.join(self.img_dir, path)
                self.samples.append((img_path, caption, class_id))

    def _extract_class_id(self, row):
        pass

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, text, class_id = self.samples[idx]

        img = self.size_transform(Image.open(img_path).convert("RGB"))
        img = _toTensor(img)

        return {
            "image": img,
            "text": text,
            "class_id": class_id,
            "idx": idx
        }


# Train SL Vision Encoder

In [ ]:
import torch
from tqdm import tqdm

In [ ]:
def sl_infonce(z_q, z_k, class_ids, temperature=0.07):
    logits = z_q @ z_k.t() / temperature
    positive_mask = class_ids.unsqueeze(0) == class_ids.unsqueeze(1)
    positive_mask = positive_mask.to(dtype=logits.dtype)

    log_denominator = torch.logsumexp(logits, dim=1)
    masked_logits = logits.masked_fill(positive_mask == 0, float('-inf'))
    log_numerator = torch.logsumexp(masked_logits, dim=1)

    valid_mask = positive_mask.sum(dim=1) > 0
    if not torch.all(valid_mask):
        log_denominator = log_denominator[valid_mask]
        log_numerator = log_numerator[valid_mask]

    loss = -(log_numerator - log_denominator)
    return loss.mean() if loss.numel() > 0 else torch.tensor(0.0, device=z_q.device)


In [ ]:
def freeze_text_encoder(model):
    if hasattr(model, "text_model"):
        for p in model.text_model.parameters():
            p.requires_grad = False


In [ ]:
def train_sl_vision_encoder(
    model,
    dataloader,
    optimizer,
    device="cuda",
    temperature=0.07,
    epochs=10,
):
    model.train()
    freeze_text_encoder(model)

    for epoch in range(epochs):
        total_loss = 0.0

        for batch in tqdm(dataloader, desc=f"Epoch {epoch+1}"):
            imgs = batch["image"].to(device)
            class_ids = batch["class_id"].to(device)

            image_feats = model.encode_posttransform_image(imgs)
            loss = sl_infonce(
                image_feats,
                image_feats,
                class_ids,
                temperature=temperature
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"[Epoch {epoch+1}] SL Loss: {total_loss/len(dataloader):.4f}")


# Config

In [ ]:
model_name = 'biomedclip'
dataset_name = 'entrep'
mode_pretrained = 'scratch'
size_transform = SIZE_TRANSFORM[model_name]
batch_size = 64
epochs = 50
temperature = 0.07

In [ ]:
from torch.utils.data import DataLoader
import torch
import yaml

if model_name in ['medclip', 'biomedclip']:
    model = ModelFactory.create_model(
        model_type=model_name,
        variant='base',
        pretrained=True,
        mode_pretrained=mode_pretrained
    )

elif model_name == 'entrep':
    config_path = "configs/entrep_contrastive.yaml"
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    model_config = config.get('model', {})
    model = ModelFactory.create_model(
        model_type="entrep",
        variant='base',
        checkpoint=None,
        pretrained=False,
        **{k: v for k, v in model_config.items() if k != 'model_type' and k != "pretrained" and k != "checkpoint"},
        mode_pretrained=mode_pretrained
    )

model = model.cuda()

if dataset_name == "mimic":
    dataset = MiMic(
        size_transform=size_transform,
    )
elif dataset_name == "entrep":
    dataset = ENTREPDataset(
        size_transform=size_transform
    )
else:
    raise ValueError(f"Unsupported dataset for this SL notebook: {dataset_name}")

sl_loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=True,
)

if model_name == "biomedclip":
    optimizer = torch.optim.AdamW(
        model.visual.parameters(),
        lr=1e-4,
        weight_decay=1e-4
    )
else:
    optimizer = torch.optim.AdamW(
        model.vision_model.parameters(),
        lr=1e-4,
        weight_decay=1e-4
    )


In [ ]:
train_sl_vision_encoder(
    model,
    sl_loader,
    optimizer,
    temperature=temperature,
    epochs=epochs
)

# Contasive Learning

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm

def clip_infonce_loss(image_feats, text_feats, temperature=0.07):

    logits = image_feats @ text_feats.t() / temperature
    labels = torch.arange(logits.size(0), device=logits.device)

    loss_t2i = F.cross_entropy(logits.t(), labels)
    loss_i2t = F.cross_entropy(logits, labels)

    return 0.5 * (loss_t2i + loss_i2t)


In [ ]:
def train_clip(
    model,
    dataloader,
    optimizer,
    device="cuda",
    epochs=10,
    temperature=0.07,
):
    model.train()

    for epoch in range(epochs):
        total_loss = 0.0

        for batch in tqdm(dataloader, desc=f"Epoch {epoch+1}"):
            imgs  = batch["image"].to(device)
            texts = batch["text"]

            # ===== forward =====
            image_feats = model.encode_posttransform_image(imgs)
            text_feats  = model.encode_text(texts)

            loss = clip_infonce_loss(
                image_feats,
                text_feats,
                temperature=temperature
            )

            # ===== backward =====
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(
            f"[Epoch {epoch+1}/{epochs}] "
            f"CLIP Loss: {total_loss/len(dataloader):.4f}"
        )


In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    drop_last=True
)

# ===== optimizer =====
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

# ===== train =====
train_clip(
    model,
    dataloader,
    optimizer,
    device="cuda",
    epochs=epochs,
    temperature=0.07
)


In [ ]:
torch.save(model.state_dict(), f'{model_name}_SL.pth')